================================================================= <br>
This notebook generates netcdf files for the following datasets: <br>
- Raw InTEM observations <br>
- InTEM observations with the InTEM baseline subtracted (= InTEM fixed) <br>
- RHIME observations with the InTEM baseline subtracted (= RHIME fixed) <br>
================================================================= <br>

Author: Helene De Longueville <br>
Project: PARIS <br>
Year: 2024

In [1]:
import xarray as xr
import glob
import os
from openghg.standardise import standardise_surface

# 1. Process intem (fixed) netcdf files 

In [ ]:
# Inputs
data_dir = '/user/work/qq24644/my_paris/InTEM_obs/raw_files/hfc143a/' # path to rao InTEM files
output_dir = '/user/work/qq24644/my_paris/InTEM_obs/intem_aligned/hfc143a/' # path to save processed InTEM files
baseline_subtracted = True # if True, subtract baseline

In [ ]:
file_list = glob.glob(os.path.join(data_dir, '*.nc'))

for file_path in file_list:
    print(f'Attempting to read data from {file_path}')

    fn = file_path.split('/')[-1]
    species = fn.split('_')[-2]    
    site = fn.split('_')[-3]

    output_path = file_path.replace(data_dir, output_dir, 1)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with xr.open_dataset(file_path) as ds:
            ds = ds
    vars_to_drop = [f'{species}_repeatability', f'{species}_variability', f'{species}_model_uncertainty']
    vars_to_drop = [var for var in vars_to_drop if var in ds]
    ds = ds.drop_vars(vars_to_drop)

    if species=='pfc218':
        species = 'c3f8'
        ds = ds.rename_vars({'pfc218': f'{species}'})
        ds = ds.rename_vars({'pfc218_total_uncertainty': f'{species}_repeatability'})
    else:
        ds = ds.rename_vars({f'{species}_total_uncertainty': f'{species}_repeatability'})
        
    ds.attrs["species"] = species
    ds.attrs["data_owner_email"] = "alistair.manning@metoffice.gov.uk"

    if baseline_subtracted is True:
        ds = ds.assign(**{species: ds[species] - ds['posterior_baseline']})

    if species == 'n2o' | species == 'ch4':
        ds[species] *= 1e9
        ds[f'{species}_repeatability'] *= 1e9
        ds['posterior_baseline'] *= 1e9

        ds[species].attrs['units'] = '1e-9 mol/mol'
        ds[f'{species}_repeatability'].attrs['units'] = '1e-9 mol/mol'
        ds['posterior_baseline'].attrs['units'] = '1e-9 mol/mol'
    else:
        ds[species] *= 1e12
        ds[f'{species}_repeatability'] *= 1e12
        ds['posterior_baseline'] *= 1e12

        ds[species].attrs['units'] = '1e-12 mol/mol'
        ds[f'{species}_repeatability'].attrs['units'] = '1e-12 mol/mol'
        ds['posterior_baseline'].attrs['units'] = '1e-12 mol/mol'

    ds.to_netcdf(output_path)

    print(f'Processed and save: {output_path}')

# 2. Process rhime fixed netcdf files

## a. Create RHIME obs files for each site from PARIS outputs

In [ ]:
# Inputs
species = 'pfc218'
rhime_path = '/group/chem/acrg/PARIS_inversions/pfc218/RHIME_NAME_EUROPE_FLAT_rhime_obs_rhime_baseline_optimized_pfc218_yearly/formatted_outputs/RHIME_NAME_EUROPE_FLAT_rhime_obs_rhime_baseline_optimized_pfc218_yearly_concentrations.nc'
output_path = f'/user/work/qq24644/my_paris/InTEM_obs/rhime_files/{species}/'

In [75]:
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with xr.open_dataset(rhime_path) as ds:
    ds = ds

vars_to_drop = ['uYmod', 'uYtotal', 'YapostBC', 'YaprioriBC', 'Yapost', 'qYapost', 'Yapriori', 'qYapriori','percentile']
ds = ds.drop_vars(vars_to_drop)

if species=='pfc218':
    species2 = 'c3f8'
else:
    species2 = species

ds = ds.rename_vars({'Yobs': f'{species2}'})
ds = ds.rename_vars({'uYobs_repeatability': f'{species2}_repeatability'})
ds = ds.rename_vars({'uYobs_variability': f'{species2}_variability'})

ds[species2] *= 1e12
ds[f'{species2}_repeatability'] *= 1e12
ds[f'{species2}_variability'] *= 1e12

ds[species2].attrs['units'] = '1e-12 mol/mol'
ds[f'{species2}_repeatability'].attrs['units'] = '1e-12 mol/mol'
ds[f'{species2}_variability'].attrs['units'] = '1e-12 mol/mol'

sites = ds.sitenames.values

for idx, site in enumerate(sites):
    ds_site = ds.sel(nsite=idx)
    ds_site = ds_site.drop_vars('sitenames')

    fn = f'RHIME_NAME_EUROPE_rhime_obs_{site}_{species}_obs.nc'
    ds_site.to_netcdf(output_path+fn)
    print(f'Created and save: {output_path+fn}')


Created and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_CMN_pfc218_obs.nc
Created and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_JFJ_pfc218_obs.nc
Created and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_MHD_pfc218_obs.nc
Created and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_TAC_pfc218_obs.nc


## b. Create RHIME obs - InTEM baseline files

In [ ]:
# Inputs
species = 'pfc218'
data_dir = '/user/work/qq24644/my_paris/InTEM_obs/rhime_files'
intem_dir = '/user/work/qq24644/my_paris/InTEM_obs/raw_files'
output_dir = '/user/work/qq24644/my_paris/InTEM_obs/rhime_fixed'

In [76]:
file_list = glob.glob(os.path.join(data_dir, f'{species}/*.nc'))

for file_path in file_list:
    print(f'Attempting to read data from {file_path}')

    fn = file_path.split('/')[-1]
    species = fn.split('_')[-2]    
    site = fn.split('_')[-3]

    if species=='pfc218':
        species2 = 'c3f8'
    else:
        species2 = species

    output_path = f'{output_dir}/{species}/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_{site}_{species}_obs.nc'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with xr.open_dataset(file_path) as rhime:
            rhime = rhime

    intem_path = glob.glob(f'{intem_dir}/{species}/*{site}*.nc')[0]
    with xr.open_dataset(intem_path) as intem:
            intem_baseline = intem['posterior_baseline'].sel(time=rhime.time)*1e12
            inlet_magl_value = intem.attrs['inlet_magl']
            
    rhime['intem_posterior_baseline'] = intem_baseline
    rhime['intem_posterior_baseline'].attrs['units'] = '1e-12 mol/mol'
    rhime['intem_posterior_baseline'].attrs['long_name'] = 'InTEM posterior baseline for PARIS'

    rhime.attrs["species"] = species2
    rhime.attrs["site_name"] = site
    rhime.attrs['inlet_magl'] = inlet_magl_value
    rhime.attrs["created_by"] = "qq24644"

    rhime = rhime.assign(**{species2: rhime[species2] - rhime['intem_posterior_baseline']})
    rhime[species2].attrs['units'] = '1e-12 mol/mol'
    rhime[species2].attrs['long_name'] = 'RHIME obs with InTEM baseline subtracted'

    rhime.attrs["data_owner"] = 'Met Office and ACRG'
    rhime.attrs["data_owner_email"] = 'qq24644@bristol.ac.uk'

    rhime.to_netcdf(output_path)

    print(f'Processed and save: {output_path}')

Attempting to read data from /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_MHD_pfc218_obs.nc
Processed and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_MHD_pfc218_obs.nc
Attempting to read data from /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_CMN_pfc218_obs.nc
Processed and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_CMN_pfc218_obs.nc
Attempting to read data from /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_JFJ_pfc218_obs.nc
Processed and save: /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_JFJ_pfc218_obs.nc
Attempting to read data from /user/work/qq24644/my_paris/InTEM_obs/rhime_files/pfc218/RHIME_NAME_EUROPE_rhime_obs_TAC_pfc218_obs.nc
Processed and save: /user/work/qq24644/my_paris/InTEM_obs/

# 3. Populate object store

In [78]:
# Inputs
instrument = 'rhime_fixed' #'rhime_fixed' 'intem_aligned'
source_format = 'openghg'
sampling_period = '4h'
store_name = 'paris_store_zarr'
calibration_scale = 'not_set'
optional_metadata={"project": "PARIS", "baseline": "InTEM"}

data_dir = f'/user/work/qq24644/my_paris/InTEM_obs/{instrument}/{species}/'

In [79]:
file_list = glob.glob(os.path.join(data_dir, '*.nc'))

for file_path in file_list:
    print(f'Attempting to standardise data from {file_path}')

    fn = file_path.split('/')[-1]
    species = fn.split('_')[-2]
    site = fn.split('_')[-3]

    with xr.open_dataset(file_path) as ds:
            ds = ds

    inlet_magl_value = ds.attrs['inlet_magl']

    if site == 'TAC':
        network = 'decc'
    else:
        network = 'agage'

    standardise_surface(filepath=file_path, source_format=source_format, network=network, site=site, instrument=instrument, inlet=inlet_magl_value, sampling_period=sampling_period, calibration_scale=calibration_scale, update_mismatch='from_source', store=store_name, if_exists='new', save_current='yes', optional_metadata=optional_metadata)
    print(f'Done!')

Attempting to standardise data from /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_CMN_pfc218_obs.nc


2025-06-30T17:46:43 INFO     INFO:openghg.store:Created new datasource with UUID                       ]8;id=245612;file:///user/home/qq24644/openghg/openghg/store/base/_base.py\_base.py]8;;\:]8;id=559312;file:///user/home/qq24644/openghg/openghg/store/base/_base.py#391\391]8;;\
                             f0111c28-722e-4696-99e9-ca46a206ec78                                                  

                    INFO     INFO:openghg.store:Completed processing:                            ]8;id=986928;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py\_obssurface.py]8;;\:]8;id=694726;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py#409\409]8;;\
                             RHIME_NAME_EUROPE_rhime_obs_intem_baseline_CMN_pfc218_obs.nc.                         

Done!
Attempting to standardise data from /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_TAC_pfc218_obs.nc


2025-06-30T17:46:44 INFO     INFO:openghg.store:Created new datasource with UUID                       ]8;id=482007;file:///user/home/qq24644/openghg/openghg/store/base/_base.py\_base.py]8;;\:]8;id=86375;file:///user/home/qq24644/openghg/openghg/store/base/_base.py#391\391]8;;\
                             e2b47841-a5bb-4ed8-b15d-b38834bbd60e                                                  

...

                    INFO     INFO:openghg.store:Completed processing:                            ]8;id=418248;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py\_obssurface.py]8;;\:]8;id=437083;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py#409\409]8;;\
                             RHIME_NAME_EUROPE_rhime_obs_intem_baseline_MHD_pfc218_obs.nc.                         

Done!
Attempting to standardise data from /user/work/qq24644/my_paris/InTEM_obs/rhime_fixed/pfc218/RHIME_NAME_EUROPE_rhime_obs_intem_baseline_JFJ_pfc218_obs.nc


                    INFO     INFO:openghg.store:Created new datasource with UUID                       ]8;id=47701;file:///user/home/qq24644/openghg/openghg/store/base/_base.py\_base.py]8;;\:]8;id=775111;file:///user/home/qq24644/openghg/openghg/store/base/_base.py#391\391]8;;\
                             af424d60-7ce1-47d5-af93-285aca33a5ff                                                  

                    INFO     INFO:openghg.store:Completed processing:                            ]8;id=625210;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py\_obssurface.py]8;;\:]8;id=43959;file:///user/home/qq24644/openghg/openghg/store/_obssurface.py#409\409]8;;\
                             RHIME_NAME_EUROPE_rhime_obs_intem_baseline_JFJ_pfc218_obs.nc.                         

Done!
